In [1]:
from pathlib import Path
rootdir = Path("../../../..").resolve()
import sys
sys.path.insert(0, str(rootdir) )
from typing import Callable, Generator

import gemmi 
import parasail
import numpy as np
from rdkit import Chem
import matplotlib.pyplot as plt

from xaidar.data.molecModels import loadPDB
from xaidar.data.molecModels import get_pdb_stats, sele_pdb, sele_Lig, get_res_CoM
from xaidar.data.molecModels import flatten_pdb, sele_AA, createPDB, get_atom_coord

/home/eoo22534/mydir/xaidar/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dls_path = Path("/dls/labxchem/data/")
ev2a_dls_path = dls_path.joinpath("lb32627/lb32627-66/")

In [3]:
from sqlalchemy import create_engine, inspect
import pandas as pd
from pathlib import Path

# Path to your .sqlite file

sqliteFilePath = ev2a_dls_path.joinpath( "processing/database/soakDBDataFile.sqlite").resolve()

print("Sqlite File Path: {}".format( sqliteFilePath ) )
# Step 1: Create an SQLAlchemy engine to connect to the SQLite database
engine = create_engine(f"sqlite:///{sqliteFilePath}")

# Step 2: Inspect the database to see available tables
inspector = inspect(engine)
tables = inspector.get_table_names()
print("Tables in the database:")
print(tables)

# Step 3: Load a specific table into a pandas DataFrame
tables = ['depositTable', 'mainTable', 'panddaTable', 'soakDB']# ['mainTable', 'panddaTable', 'collectionTable', 'depositTable','soakDB', 'zenodoTable','Pucks', ]
loadtable = lambda table_name : pd.read_sql(f"SELECT * FROM {table_name}", con=engine)
tabledf_dict = {table_name : loadtable(table_name) for table_name in tables[:4]  }

# for df in tabledf_dict.values():
#     filter_colms = [ colm for colm in df.columns if not all( df[colm].isna() ) ]    # Show columns with data
#     display( filter_colms)
    # display( df] )
# tabledf_lst = maindf, panddadf, collectdf, depoistdf 
# print(type( tabledf_dict.values()))

# display( list( tabledf_dict.values() )[0].iloc[:,:3])



# Step 6: Close the connection (optional, as SQLAlchemy handles it automatically)
engine.dispose()

Sqlite File Path: /dls/labxchem/data/lb32627/lb32627-66/processing/database/soakDBDataFile.sqlite
Tables in the database:
['Pucks', 'collectionTable', 'depositTable', 'labelTable', 'mainTable', 'panddaTable', 'soakDB', 'zenodoTable']


In [4]:
display( tabledf_dict["mainTable"].head() )

,ID,LabVisit,LibraryPlate,SourceWell,LibraryName,CompoundSMILES,CompoundCode,CrystalPlate,CrystalWell,EchoX,...,Deposition_PDB_ID,Deposition_PDB_file,Deposition_Date,Deposition_mmCIF_model_file,Deposition_mmCIF_SF_file,Label,table_one,AssayIC50,LastUpdated,LastUpdated_by
0,2,lb32627-66,Diffraction Test,None,None,CS(=O)C CS(C)=O,DMSO,9bjs_2023-05-15_RI1000-0276-3drop,H12c,0.0,...,None,None,None,None,None,None,None,None,13/10/2023 10:29,vfi61159
1,3,lb32627-66,Diffraction Test,None,None,CS(=O)C CS(C)=O,DMSO,9blm_2023-05-22_RI1000-0276-3drop,B02a,0.0,...,None,None,None,None,None,None,None,None,04/07/2025 14:41,ill13029
2,4,lb32627-66,Diffraction Test,None,None,CS(=O)C CS(C)=O,DMSO,9blm_2023-05-22_RI1000-0276-3drop,B11a,0.0,...,None,None,None,None,None,None,None,None,12/03/2024 11:04,vfi61159
3,5,lb32627-66,Diffraction Test,None,None,CS(=O)C CS(C)=O,DMSO,9blm_2023-05-22_RI1000-0276-3drop,F06a,0.0,...,None,None,None,None,None,None,None,None,12/03/2024 11:04,vfi61159
4,6,lb32627-66,Diffraction Test,None,None,CS(=O)C CS(C)=O,DMSO,9blm_2023-05-22_RI1000-0276-3drop,H01a,0.0,...,None,None,None,None,None,None,None,None,12/03/2024 11:04,vfi61159


In [5]:
main_df = tabledf_dict["mainTable"]
pandda_df = tabledf_dict["panddaTable"]
# collect_df = tabledf_dict["collectionTable"]

## Main Table

In [6]:
column_of_interest = [
    "ProteinName",
    "CrystalName",
    "LibraryName",
    "DimpleReferencePDB",
    "DimplePathToMTZ",
    "DimplePathToPDB",
    "DimplePANDDApath",
    "RefinementCIF",
    "RefinementPDB_latest",
    "RefinementMTZ_latest",
    "RefinementMMCIFmodel_latest",
    "RefinementMMCIFreflections_latest",
    "DataProcessingResolutionHigh",
    "DataProcessingResolutionLow",
    "DataProcessingRmergeOverall",
    "DataProcessingRmergeHigh",
    "DataProcessingRmergeLow",
    "DataProcessingIsigOverall",
    "DataProcessingIsigHigh",
    "DataProcessingIsigLow",
    "DataProcessingCompletenessOverall",
    "DataProcessingCompletenessHigh",
    "DataProcessingCompletenessLow",
    "DataProcessingCChalfOverall",
    "DataProcessingCChalfHigh",
    "DataProcessingCChalfLow",
    "DimpleRcryst",
    "DimpleRfree"
]

In [7]:
display(main_df[column_of_interest].head() )

,ProteinName,CrystalName,LibraryName,DimpleReferencePDB,DimplePathToMTZ,DimplePathToPDB,DimplePANDDApath,RefinementCIF,RefinementPDB_latest,RefinementMTZ_latest,...,DataProcessingIsigHigh,DataProcessingIsigLow,DataProcessingCompletenessOverall,DataProcessingCompletenessHigh,DataProcessingCompletenessLow,DataProcessingCChalfOverall,DataProcessingCChalfHigh,DataProcessingCChalfLow,DimpleRcryst,DimpleRfree
0,A71EV2A,A71EV2A-x0003,None,None,None,None,None,/dls/labxchem/data/lb32627/lb32627-66/processi...,None,None,...,0.8,13,100,100,99.8,0.978,0.353,0.967,None,None
1,A71EV2A,A71EV2A-x0004,None,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,...,0.7,35.6,99.6,92.6,100,0.94,0.53,0.96,0.26231,0.34009
2,A71EV2A,A71EV2A-x0005,None,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,None,...,-0.9,19.1,76.3,34.6,82.4,0.55,0.29,0.85,0.50893,0.54378
3,A71EV2A,A71EV2A-x0006,None,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,None,None,...,0.1,29.4,92,89.1,97.2,0.79,0.72,0.91,0.35612,0.39663
4,A71EV2A,A71EV2A-x0007,None,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,None,None,...,0.3,2.8,70.3,27.9,76.3,0.36,0.01,0.67,0.59809,0.62719


In [8]:
df_filter = (

(main_df[ "DataProcessingDimpleSuccessful" ] == "TRUE" ) &                         # Managed to be processed by dimple
(main_df[ "DimpleStatus" ] == "finished" ) &

(main_df[ "DimplePANDDAhit"] != "TRUE" )  &                                        # no Hit

(main_df[ "RefinementOutcome" ] == "6 - Deposited") &
(main_df[ "RefinementStatus" ] == "finished")

)
print( df_filter.value_counts() )


False    8418
True      365
Name: count, dtype: int64


In [9]:
from IPython.display import display
display(main_df[column_of_interest][df_filter])
filter_df = main_df[column_of_interest][df_filter]

,ProteinName,CrystalName,LibraryName,DimpleReferencePDB,DimplePathToMTZ,DimplePathToPDB,DimplePANDDApath,RefinementCIF,RefinementPDB_latest,RefinementMTZ_latest,...,DataProcessingIsigHigh,DataProcessingIsigLow,DataProcessingCompletenessOverall,DataProcessingCompletenessHigh,DataProcessingCompletenessLow,DataProcessingCChalfOverall,DataProcessingCChalfHigh,DataProcessingCChalfLow,DimpleRcryst,DimpleRfree
3564,A71EV2A,A71EV2A-x2972,ASAPPTBOAM,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,...,1.1,16.1,98.1,98.8,98.2,0.992,0.355,0.997,0.29187,0.33483
3647,A71EV2A,A71EV2A-x3054,ASAPPTBOAM,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,...,1.3,30.5,100,100,99.5,0.995,0.399,0.99,0.2966,0.35304
3776,A71EV2A,A71EV2A-x3175,ASAPPTBOAM,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,...,0.7,14.4,99.9,99.8,100,0.995,0.366,0.997,0.27691,0.29693
3777,A71EV2A,A71EV2A-x3176,ASAPPTBOAM,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,...,0.2,22,99.4,92.1,100,0.92,0.15,0.99,0.30382,0.33895
3778,A71EV2A,A71EV2A-x3177,ASAPPTBOAM,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,...,0.6,24.7,99.9,100,99.9,0.989,0.376,0.986,0.29791,0.32252
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6088,A71EV2A,A71EV2A-x5289,ASAPPTBOAM,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,None,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,...,1.3,22.7,99.7,98.2,98.8,0.995,0.347,0.992,0.19937,0.22585
6120,A71EV2A,A71EV2A-x5317,ASAPPTBOAM,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,None,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,...,0.8,89.9,96.3,67.7,100.0,1.0,0.28,1.0,0.24211,0.25921
6142,A71EV2A,A71EV2A-x5339,ASAPPTBOAM,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,None,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,/dls/labxchem/data/lb32627/lb32627-66/processi...,...,1.2,15.0,100.0,99.9,99.8,0.995,0.491,0.992,0.20394,0.2392
6163,A71EV2A,A71EV2A-x5359,ASAPPTBOAM,/dls/labxchem/data